In [131]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [132]:
train = pd.read_csv('../data/raw/raw_eda_hr_train.csv')
test = pd.read_csv('../data/raw/raw_eda_hr_test.csv')

X_train = train.drop('Attrition', axis=1)
y_train = train['Attrition']
X_test = test.drop('Attrition', axis=1)
y_test = test['Attrition']

print(f"Train: {X_train.shape[0]} (positive: {y_train.sum()})")
print(f"Test: {X_test.shape[0]} (positive: {y_test.sum()})")

Train: 1176 (positive: 190)
Test: 294 (positive: 47)


In [133]:
def simple_eng(df):
    df = df.copy()
    drop_cols = ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours',
                 'DailyRate', 'MonthlyRate', 'StockOptionLevel', 'PercentSalaryHike']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
    df['OverTime'] = (df['OverTime'] == 'Yes').astype(int)
    df['IsSingle'] = (df['MaritalStatus'] == 'Single').astype(int)
    df['Travel_Rarely'] = (df['BusinessTravel'] == 'Travel_Rarely').astype(int)
    df['Travel_Frequently'] = (df['BusinessTravel'] == 'Travel_Frequently').astype(int)
    df['tenure_ratio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1e-5)
    df['satisfaction'] = (df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction']) / 3
    df.drop(columns=['MaritalStatus', 'BusinessTravel', 'Gender'], inplace=True, errors='ignore')
    return df

X_train = simple_eng(X_train)
X_test = simple_eng(X_test)

In [134]:
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print("Categorical:", cat_cols)
print("Numerical count:", len(num_cols))

Categorical: ['Department', 'EducationField', 'JobRole']
Numerical count: 25


In [135]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols)
])

In [136]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe = ImbPipeline([
    ('prep', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
])
param_grid = {'clf__C': [0.01, 0.05, 0.1, 0.5, 1.0]}
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)
best_model = grid.best_estimator_
print(f"Best C: {grid.best_params_['clf__C']}, best CV recall: {grid.best_score_:.4f}")

Best C: 0.01, best CV recall: 0.7263


In [137]:
probs = best_model.predict_proba(X_test)[:, 1]

In [138]:
thresholds = np.arange(0.1, 0.9, 0.02)
best_thresh = 0.5
best_f1 = 0
for thresh in thresholds:
    y_pred = (probs >= thresh).astype(int)
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh
print(f"Optimal threshold by F1: {best_thresh:.3f} (F1={best_f1:.4f})")

Optimal threshold by F1: 0.720 (F1=0.6000)


In [139]:
final_thresh = 0.66  
y_pred_final = (probs >= final_thresh).astype(int)

In [140]:
accuracy = accuracy_score(y_test, y_pred_final)
precision = precision_score(y_test, y_pred_final)
recall = recall_score(y_test, y_pred_final)
f1 = f1_score(y_test, y_pred_final)
roc_auc = roc_auc_score(y_test, probs)

print("\n" + "="*65)
print("Финальные метрики модели (LogisticRegression + SMOTE, порог={})".format(final_thresh))
print("="*65)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_final))


Финальные метрики модели (LogisticRegression + SMOTE, порог=0.66)
Accuracy:  0.8605
Precision: 0.5652
Recall:    0.5532
F1-score:  0.5591
ROC-AUC:   0.8195

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.92      0.92       247
           1       0.57      0.55      0.56        47

    accuracy                           0.86       294
   macro avg       0.74      0.74      0.74       294
weighted avg       0.86      0.86      0.86       294

